# Diversity, Cold Start & Exploration

Companion notebook for the [Diversity, Cold Start & Exploration lesson](https://ml-viz-ruby.vercel.app/courses/recommender-systems/05-diversity-cold-start-exploration).

We implement MMR diversification, content-based cold start, and Thompson Sampling exploration. Pure NumPy.

> **To save your work:** File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('dark_background')
rng = np.random.default_rng(0)

## 1 — Maximal Marginal Relevance (MMR)

MMR greedily selects items that are both relevant and diverse. At each step it balances the relevance score with similarity to already-selected items.

In [ ]:
def mmr(scores, emb, k, lam=0.5):
    """
    scores: (N,) relevance scores
    emb:    (N, d) item embeddings
    k:      number of items to select
    lam:    trade-off (1 = pure relevance, 0 = pure diversity)
    Returns indices of k selected items.
    """
    N = len(scores)
    selected, remaining = [], list(range(N))
    for _ in range(k):
        if not selected:
            # First item: pick highest relevance
            best = max(remaining, key=lambda i: scores[i])
        else:
            S_emb = emb[selected]                       # (|S|, d)
            def mmr_score(i):
                rel = scores[i]
                sim = float((emb[i] @ S_emb.T).max())  # max similarity to selected
                return lam * rel - (1 - lam) * sim
            best = max(remaining, key=mmr_score)
        selected.append(best)
        remaining.remove(best)
    return selected

# Create synthetic items: 50 items with d=8 embeddings and relevance scores
n, d = 50, 8
item_emb = rng.normal(size=(n, d))
item_emb /= np.linalg.norm(item_emb, axis=1, keepdims=True)
relevance = rng.uniform(0, 1, n)  # simulated relevance scores

top5_relevant = np.argsort(-relevance)[:5].tolist()
top5_diverse = mmr(relevance, item_emb, k=5, lam=0.5)

print("Top-5 by relevance only:", top5_relevant)
print("Top-5 by MMR (λ=0.5):  ", top5_diverse)

## 2 — Content-based cold start

For a new item with no interactions, we project its content features into the item embedding space using a learned projection.

In [ ]:
# Simulate a content-feature -> embedding projection (pre-trained)
d_content, d_emb = 16, 8
W_proj = rng.normal(size=(d_emb, d_content)) * 0.1    # projection matrix

def cold_start_embedding(content_features):
    """Map content features to the shared embedding space."""
    emb = W_proj @ content_features
    return emb / np.linalg.norm(emb)

# New item: content features (e.g., text/image features)
new_item_content = rng.normal(size=d_content)
new_item_emb = cold_start_embedding(new_item_content)

# Find most similar existing items by cosine similarity
sims = item_emb @ new_item_emb
top3_similar = np.argsort(-sims)[:3]
print("New item embedding:", new_item_emb.round(3))
print("Most similar existing items:", top3_similar.tolist())
print("Similarity scores:", sims[top3_similar].round(3))

## 3 — Thompson Sampling

Thompson Sampling maintains a Beta posterior over each arm's click probability and samples from it to decide which item to recommend.

In [ ]:
class ThompsonSamplingBandit:
    def __init__(self, n_arms):
        self.alpha = np.ones(n_arms)   # successes + 1
        self.beta  = np.ones(n_arms)   # failures + 1

    def select_arm(self, rng_):
        samples = rng_.beta(self.alpha, self.beta)
        return int(np.argmax(samples))

    def update(self, arm, reward):
        self.alpha[arm] += reward
        self.beta[arm]  += (1 - reward)

# Simulate 200 rounds: 5 items with true click rates
n_arms = 5
true_rates = np.array([0.05, 0.15, 0.40, 0.20, 0.10])  # item 2 is best
bandit = ThompsonSamplingBandit(n_arms)

arm_counts = np.zeros(n_arms, dtype=int)
for t in range(200):
    arm = bandit.select_arm(rng)
    reward = float(rng.random() < true_rates[arm])
    bandit.update(arm, reward)
    arm_counts[arm] += 1

print("Arms pulled:", arm_counts)
print("Best arm (true):", true_rates.argmax(), f"(rate={true_rates.max():.2f})")
print("Most pulled arm:", arm_counts.argmax())

## ✏️ Your turn

**Exercise.** Implement `compute_ild(selected_emb)` — the Intra-List Diversity (ILD) metric: the average pairwise *cosine distance* (1 − cosine similarity) between all pairs of selected items.

Compare the ILD of the relevance-only top-5 selection vs. the MMR top-5 selection.

In [ ]:
def compute_ild(selected_emb):
    """
    selected_emb: (k, d) embeddings of selected items
    Returns: mean pairwise cosine distance (scalar)
    """
    # TODO(you): compute mean pairwise (1 - cosine_similarity) for all pairs i ≠ j
    return ...

ild_relevant = compute_ild(item_emb[top5_relevant])
ild_diverse  = compute_ild(item_emb[top5_diverse])
print(f"ILD (relevance-only): {ild_relevant:.4f}")
print(f"ILD (MMR λ=0.5):      {ild_diverse:.4f}")
print("MMR should be more diverse (higher ILD):", ild_diverse > ild_relevant)

In [ ]:
# Assertion cell
def _ref_ild(E):
    k = len(E)
    sims = E @ E.T
    total = sum(1 - sims[i,j] for i in range(k) for j in range(k) if i != j)
    return total / (k*(k-1))
assert abs(compute_ild(item_emb[top5_relevant]) - _ref_ild(item_emb[top5_relevant])) < 1e-6
print("✓ ILD implementation is correct")

<details>
<summary>Solution</summary>

```python
def compute_ild(selected_emb):
    k = len(selected_emb)
    sims = selected_emb @ selected_emb.T   # (k, k) cosine similarities
    distances = 1 - sims
    # sum all off-diagonal entries, divide by k*(k-1)
    return (distances.sum() - np.trace(distances)) / (k * (k - 1))
```
</details>